# LoRA Maker — SDXL LoRA Training

Kaggle notebook for rebuilding the SDXL LoRA training workflow from scratch. The only project dataset input is `/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset`.

**Before running:** enable a GPU in Kaggle settings and enable Internet so the notebook can fetch the repository script, dependencies, and base model. Training has not been pre-run or verified.

In [ ]:
from pathlib import Path
import os

DATASET = Path('/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset')
OUTPUT = Path('/kaggle/working/lora-maker-output')
MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
INSTANCE_PROMPT = 'a photo of yamazaki shishi'  # Adjust to accurately describe the dataset
MAX_TRAIN_STEPS = 1000
RANK = 16
RESOLUTION = 1024

assert DATASET.is_dir(), f'Dataset not found: {DATASET}. Attach the Kaggle dataset first.'
print('Dataset:', DATASET)
print('Output:', OUTPUT)
print('Image files:', sum(1 for p in DATASET.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}))

## 1. Check the runtime

This workflow requires a CUDA-enabled Kaggle GPU for training.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU under Kaggle Notebook Settings > Accelerator, then rerun.')

## 2. Install training dependencies

The install step intentionally does not replace PyTorch, preserving the Kaggle CUDA build.

In [ ]:
%pip install -U diffusers transformers accelerate peft safetensors

## 3. Fetch the repository training script

This downloads the Python implementation from the repository root so the notebook and CLI share one implementation.

In [ ]:
import urllib.request

SCRIPT_URL = 'https://raw.githubusercontent.com/bonsai/lora-maker/main/lora-maker.py'
SCRIPT_PATH = Path('/kaggle/working/lora-maker.py')
request = urllib.request.Request(SCRIPT_URL, headers={'User-Agent': 'bonsai-lora-maker-notebook/1.0'})
with urllib.request.urlopen(request, timeout=60) as response:
    script_bytes = response.read()
if b'DEFAULT_DATASET' not in script_bytes or b'train_dreambooth_lora_sdxl.py' not in script_bytes:
    raise RuntimeError('Downloaded file did not pass the script sanity check.')
SCRIPT_PATH.write_bytes(script_bytes)
print('Saved:', SCRIPT_PATH, '| bytes:', len(script_bytes))

## 4. Run training

The command copies supported images to a working directory, downloads the Diffusers SDXL DreamBooth LoRA example, runs training, and checks for adapter weights. The first run downloads the SDXL base model and may take a long time.

In [ ]:
import subprocess, sys

command = [
    sys.executable, str(SCRIPT_PATH),
    '--dataset', str(DATASET),
    '--output', str(OUTPUT),
    '--model', MODEL,
    '--instance-prompt', INSTANCE_PROMPT,
    '--resolution', str(RESOLUTION),
    '--rank', str(RANK),
    '--max-train-steps', str(MAX_TRAIN_STEPS),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

## 5. Verify adapter artifacts

A successful notebook run should produce trained LoRA weights under the output directory. A downloaded base model alone does not count as success.

In [ ]:
adapter_files = [p for p in OUTPUT.rglob('*') if p.is_file() and p.name in {'pytorch_lora_weights.safetensors', 'pytorch_lora_weights.bin'}]
if not adapter_files:
    raise FileNotFoundError(f'No LoRA adapter weights found under {OUTPUT}. Inspect training logs.')
for path in adapter_files:
    print(f'{path} — {path.stat().st_size:,} bytes')
print('Adapter artifact verification passed. Run an inference test before treating the model as production-ready.')